# Movie Poster ROI Prediction — ResNet50

Trains a ResNet50 + MLP regression model to predict inflation-adjusted ROI
from movie poster images alone.

**Pipeline:**
1. Load and clean `master_movie_data.csv` (fix known budget errors, validate poster paths)
2. Build a `PosterOnlyDataset` that serves poster images with `log1p(roi_2025)` targets
3. Extract image features from a pretrained ResNet50 (optionally fine-tuning `layer4`)
4. Train a 3-layer MLP with Huber loss + optional diversity penalty
5. Evaluate on the held-out test set and visualize predictions with Grad-CAM

## Installation

In [ ]:
!pip install torch torchvision torchsummary pandas pillow scikit-learn tqdm

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from scipy import stats
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import time
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## Data Preparation

`prepare_data` performs the following steps on `master_movie_data.csv`:
- Corrects three known budget data entry errors using hard-coded ground-truth values
- Validates and repairs poster file paths (attempts to remap to `POSTER_DIR` if stale)
- Drops rows without a valid `roi_2025` or a poster file on disk
- Deduplicates by `poster_file` to prevent data leakage between splits
- Saves the cleaned data back to the source CSV, then produces train/test CSVs

In [ ]:
BASE_DIR   = Path('/content/gdrive/Shareddrives/FML_FINAL/Data')
POSTER_DIR = BASE_DIR / 'posters'
RAW_CSV    = BASE_DIR / 'master_movie_data.csv'


def prepare_data():
    print('\n--- Data Preparation ---')
    df = pd.read_csv(RAW_CSV)
    print(f'Raw rows: {len(df):,}')

    # Correct known budget data entry errors.
    # Each tuple: (tmdb_id, erroneous_value, correct_value, title)
    print('Correcting known budget data errors...')
    budget_fixes = [
        (345,   67,      65_000_000, 'Eyes Wide Shut'),
        (48787, 2,        2_000_000, 'Mute Witness'),
        (539,   1999,      806_947,  'Psycho'),
    ]
    for tmdb_id, _old_val, new_val, title in budget_fixes:
        mask = df['tmdb_id'] == tmdb_id
        if mask.any():
            old_raw    = df.loc[mask, 'budget_raw'].iloc[0]
            old_adj    = df.loc[mask, 'budget_2025'].iloc[0]
            multiplier = old_adj / old_raw if old_raw > 0 else 1.0
            new_adj    = new_val * multiplier
            df.loc[mask, 'budget_raw']   = new_val
            df.loc[mask, 'budget_2025']  = new_adj
            gross_2025 = df.loc[mask, 'gross_2025'].iloc[0]
            new_roi    = (gross_2025 - new_adj) / new_adj
            df.loc[mask, 'roi_2025']     = new_roi
            df.loc[mask, 'log_roi_2025'] = np.log1p(max(0, new_roi))
            print(f'  Fixed {title}')

    df = df.dropna(subset=['roi_2025'])
    print(f'After dropping missing roi_2025: {len(df):,}')

    # Validate poster file paths; attempt to remap stale paths to POSTER_DIR.
    valid_paths  = df['poster_file'].notna()
    exists_mask  = pd.Series(False, index=df.index)
    for idx, path in df.loc[valid_paths, 'poster_file'].items():
        try:
            exists_mask.loc[idx] = Path(path).exists()
        except Exception:
            exists_mask.loc[idx] = False

    # print(f'  {exists_mask.sum():,} paths already valid')  # debug

    broken_mask  = valid_paths & ~exists_mask
    broken_count = broken_mask.sum()
    if broken_count > 0:
        # print(f'  Attempting to fix {broken_count:,} broken paths...')  # debug
        df.loc[broken_mask, 'poster_file'] = df.loc[broken_mask, 'poster_file'].apply(
            lambda p: str(POSTER_DIR / Path(p).name)
        )
        for idx, path in df.loc[broken_mask, 'poster_file'].items():
            try:
                exists_mask.loc[idx] = Path(path).exists()
            except Exception:
                exists_mask.loc[idx] = False
        # print(f'  Fixed {exists_mask.loc[broken_mask].sum():,} paths')  # debug

    df      = df[exists_mask]
    dropped = len(valid_paths) - len(df)
    if dropped > 0:
        print(f'  Dropped {dropped:,} rows with missing or unreadable posters')

    # print(f'After poster validation: {len(df):,} rows')  # debug

    before_dedup   = len(df)
    df             = df.drop_duplicates(subset=['poster_file'], keep='first')
    dedup_removed  = before_dedup - len(df)
    if dedup_removed > 0:
        print(f'  Removed {dedup_removed:,} duplicate poster entries')

    # print(f'After deduplication: {len(df):,} rows with unique posters')  # debug

    # # Diagnostic: verify no duplicates remain
    # dup_check = df[df.duplicated(subset=['poster_file'], keep=False)]
    # if len(dup_check) > 0:
    #     print(f'  WARNING: {len(dup_check)} duplicates still present')
    #     print(dup_check[['tmdb_id', 'title', 'poster_file']].head())

    df.to_csv(RAW_CSV, index=False)

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    train_df.to_csv(BASE_DIR / 'master_movie_train.csv', index=False)
    test_df.to_csv( BASE_DIR / 'master_movie_test.csv',  index=False)

    print(f'  Saved master_movie_train.csv ({len(train_df):,} rows)')
    print(f'  Saved master_movie_test.csv  ({len(test_df):,} rows)')
    return train_df, test_df

## Model Configuration

All hyperparameters are defined here so they can be adjusted without re-running
the model definition cells.

In [ ]:
TRAIN_CSV  = BASE_DIR / 'master_movie_train.csv'
TEST_CSV   = BASE_DIR / 'master_movie_test.csv'

TARGET_COL       = 'roi_2025'
USE_LOG_TARGET   = True         # apply log1p to target before training
N_EPOCHS         = 50
BATCH_SIZE       = 32
LR               = 5e-4
NEURONS          = 512          # width of the first MLP hidden layer
LOSS_FN          = nn.HuberLoss(delta=1.0)

USE_DIVERSITY_LOSS = True       # penalizes collapse toward the mean prediction
DIVERSITY_WEIGHT   = 0.05

FINETUNE_RESNET  = True         # unfreeze ResNet layer4 during feature extraction

## Dataset, Feature Extraction & Model

- **`PosterOnlyDataset`** — loads poster images and `roi_2025` (optionally log-transformed).
- **`AddGaussianNoise`** — lightweight augmentation applied at training time.
- **`extract_features`** — runs the frozen ResNet backbone over a DataLoader and caches
  the 2048-d feature vectors so the MLP trains on pre-extracted embeddings.
- **`build_regression_mlp`** — three-layer MLP (`NEURONS` → 256 → 128 → 1) with dropout.
- **`train_model`** — Adam optimizer, Huber loss + optional diversity penalty,
  with early stopping on validation MAE.

In [ ]:
class PosterOnlyDataset(Dataset):
    """Serves poster images paired with log1p(roi_2025) regression targets."""

    def __init__(self, csv_path: Path, transform=None,
                 target_col: str = 'roi_2025', use_log: bool = True):
        df = pd.read_csv(csv_path)
        # Remap any paths that still reference the old /content/drive mount point.
        df['poster_file'] = df['poster_file'].apply(
            lambda p: p.replace('/content/drive/', '/content/gdrive/') if isinstance(p, str) else p
        )
        df = df.dropna(subset=[target_col])
        df = df[df['poster_file'].apply(
            lambda p: isinstance(p, str) and Path(p).exists()
        )].reset_index(drop=True)

        self.paths     = df['poster_file'].tolist()
        self.titles    = df['title'].tolist()
        self.transform = transform

        raw_labels   = df[target_col].astype(float)
        self.labels  = np.log1p(raw_labels.values).astype(np.float32) if use_log else raw_labels.values.astype(np.float32)

        print(f'  Loaded {len(self.paths):,} samples from {csv_path.name}')
        print(f'  Target: {target_col} (log={use_log})')
        print(f'  ROI range: {raw_labels.min():.2f} to {raw_labels.max():.2f}  |  mean: {raw_labels.mean():.2f}')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


class AddGaussianNoise:
    """Adds Gaussian noise to a tensor image; used as a training augmentation."""

    def __init__(self, mean: float = 0.0, std: float = 0.01):
        self.mean = mean
        self.std  = std

    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean


@torch.no_grad()
def extract_features(dl: DataLoader, backbone: nn.Module):
    """Runs backbone over all batches and returns (features, labels) as CPU tensors."""
    backbone.eval()
    # t0 = time.time()  # debug timing
    all_features, all_labels = [], []
    for x, y in dl:
        all_features.append(torch.flatten(backbone(x.to(device)), start_dim=1).cpu())
        all_labels.append(y)
    # print(f'  Feature extraction: {time.time() - t0:.1f}s')  # debug timing
    return torch.cat(all_features), torch.cat(all_labels)


def build_regression_mlp(input_dim: int) -> nn.Module:
    """Returns a 3-layer MLP (NEURONS -> 256 -> 128 -> 1) with dropout."""
    return nn.Sequential(
        nn.Linear(input_dim, NEURONS), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(NEURONS, 256),       nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256, 128),           nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 1),
    ).to(device)


def train_batch(x, y, model, opt, loss_fn):
    """Single gradient step; adds diversity penalty when USE_DIVERSITY_LOSS is True."""
    model.train()
    opt.zero_grad()
    preds      = model(x).squeeze(1)
    batch_loss = loss_fn(preds, y)
    if USE_DIVERSITY_LOSS:
        # Penalise collapse toward the mean by encouraging prediction spread.
        diversity_loss = -DIVERSITY_WEIGHT * (preds.std() / (y.std() + 1e-6))
        batch_loss     = batch_loss + diversity_loss
    batch_loss.backward()
    opt.step()
    return batch_loss.detach().cpu(), preds.detach().cpu()


@torch.no_grad()
def evaluate(dl: DataLoader, model: nn.Module, loss_fn) -> tuple:
    """Returns (mean_loss, MAE, RMSE, R²) over the given DataLoader."""
    model.eval()
    all_preds, all_labels, losses = [], [], []
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        preds = model(x).squeeze(1)
        losses.append(loss_fn(preds, y).item())
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())
    preds_t, labels_t = torch.cat(all_preds), torch.cat(all_labels)
    mae   = (preds_t - labels_t).abs().mean().item()
    rmse  = ((preds_t - labels_t) ** 2).mean().sqrt().item()
    ss_res = ((labels_t - preds_t) ** 2).sum().item()
    ss_tot = ((labels_t - labels_t.mean()) ** 2).sum().item()
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return float(np.mean(losses)), mae, rmse, r2


def train_model(model: nn.Module, train_dl: DataLoader, val_dl: DataLoader = None) -> dict:
    """Trains the MLP with Adam and early stopping. Returns per-epoch history dict."""
    opt     = Adam(model.parameters(), lr=LR)
    history = {'loss': [], 'mae': [], 'val_loss': [], 'val_mae': [], 'val_r2': []}

    # # Debug: log hyperparameter configuration
    # print(f'  Model: {NEURONS} -> 256 -> 128 -> 1')
    # print(f'  Dropout: 0.5, 0.4, 0.3  |  LR: {LR}')
    # print(f'  Diversity loss: {"ON" if USE_DIVERSITY_LOSS else "OFF"} (w={DIVERSITY_WEIGHT})')
    # print(f'  ResNet fine-tuning: {"ON (layer4)" if FINETUNE_RESNET else "OFF"}')

    best_val_mae     = float('inf')
    patience         = 10
    patience_counter = 0
    best_state       = None
    t0               = time.time()

    for epoch in tqdm(range(N_EPOCHS), desc='Training'):
        epoch_losses, epoch_preds, epoch_labels = [], [], []
        for x, y in train_dl:
            x, y  = x.to(device), y.to(device)
            loss, preds = train_batch(x, y, model, opt, LOSS_FN)
            epoch_losses.append(loss.item())
            epoch_preds.append(preds)
            epoch_labels.append(y.cpu())
        train_loss = float(np.mean(epoch_losses))
        train_mae  = (torch.cat(epoch_preds) - torch.cat(epoch_labels)).abs().mean().item()
        history['loss'].append(train_loss)
        history['mae'].append(train_mae)
        log = f'  Epoch {epoch+1:02d}/{N_EPOCHS} | Loss: {train_loss:.4f} | MAE: {train_mae:.4f}'

        if val_dl:
            vl, vmae, vrmse, vr2 = evaluate(val_dl, model, LOSS_FN)
            history['val_loss'].append(vl)
            history['val_mae'].append(vmae)
            history['val_r2'].append(vr2)
            log += f' || Val MAE: {vmae:.4f} | Val R2: {vr2:.4f}'
            if vmae < best_val_mae:
                best_val_mae     = vmae
                patience_counter = 0
                best_state       = model.state_dict().copy()
                log             += ' *'
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)')
                print(f'  Restoring best model (Val MAE: {best_val_mae:.4f})')
                model.load_state_dict(best_state)
                break
        print(log)

    print(f'  Training complete in {time.time() - t0:.1f}s  |  Best Val MAE: {best_val_mae:.4f}')
    return history


def visualize_training(history: dict):
    """Plots loss, MAE, and (if validation data exists) R² curves."""
    epochs  = np.arange(1, len(history['loss']) + 1)
    has_val = len(history['val_loss']) > 0
    scale   = 'log1p(ROI)' if USE_LOG_TARGET else 'ROI'
    fig, axes = plt.subplots(1, 3 if has_val else 2, figsize=(15, 4))

    axes[0].set_title('Loss (Huber + Diversity)')
    axes[0].plot(epochs, history['loss'], label='train')
    if has_val: axes[0].plot(epochs, history['val_loss'], label='val')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].set_title(f'MAE ({scale})')
    axes[1].plot(epochs, history['mae'], label='train')
    if has_val: axes[1].plot(epochs, history['val_mae'], label='val')
    axes[1].set_xlabel('Epoch'); axes[1].legend()

    if has_val:
        axes[2].set_title('R² (val)')
        axes[2].plot(epochs, history['val_r2'], color='green')
        axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.8)
        axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.show()


def run_training_pipeline():
    """
    Builds ResNet50 backbone, creates datasets and DataLoaders, extracts features,
    trains the MLP, and reports final test metrics.
    Returns (mlp, resnet, test_dataset, history).
    """
    resnet    = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Identity()
    resnet    = resnet.to(device)

    if FINETUNE_RESNET:
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name
        # print('ResNet layer4 unfrozen for fine-tuning')  # debug
    else:
        for p in resnet.parameters():
            p.requires_grad = False
        # print('ResNet frozen (feature extraction only)')  # debug

    base_transform  = models.ResNet50_Weights.IMAGENET1K_V1.transforms()
    # print('Note: training uses minimal augmentation (vertical flip + Gaussian noise)')  # debug
    train_transform = transforms.Compose([
        transforms.RandomVerticalFlip(p=0.5),
        base_transform,
        AddGaussianNoise(mean=0.0, std=0.01),
    ])

    full_train_ds = PosterOnlyDataset(TRAIN_CSV, transform=train_transform,
                                      target_col=TARGET_COL, use_log=USE_LOG_TARGET)
    test_ds       = PosterOnlyDataset(TEST_CSV,  transform=base_transform,
                                      target_col=TARGET_COL, use_log=USE_LOG_TARGET)

    v_size   = int(0.2 * len(full_train_ds))
    t_size   = len(full_train_ds) - v_size
    train_ds, val_ds = torch.utils.data.random_split(full_train_ds, [t_size, v_size])

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    train_f, train_l = extract_features(train_dl, resnet)
    val_f,   val_l   = extract_features(val_dl,   resnet)
    test_f,  test_l  = extract_features(test_dl,  resnet)

    train_feat_dl = DataLoader(TensorDataset(train_f, train_l), batch_size=BATCH_SIZE, shuffle=True)
    val_feat_dl   = DataLoader(TensorDataset(val_f,   val_l),   batch_size=BATCH_SIZE, shuffle=False)
    test_feat_dl  = DataLoader(TensorDataset(test_f,  test_l),  batch_size=BATCH_SIZE, shuffle=False)

    mlp     = build_regression_mlp(train_f.shape[1])
    history = train_model(mlp, train_feat_dl, val_dl=val_feat_dl)

    test_loss, test_mae, test_rmse, test_r2 = evaluate(test_feat_dl, mlp, LOSS_FN)
    print(f'\n{"="*60}')
    print(f'FINAL TEST RESULTS')
    print(f'{"="*60}')
    print(f'  Test MAE  (log scale): {test_mae:.4f}')
    if USE_LOG_TARGET:
        print(f'  Approx ROI error:      {np.expm1(test_mae):.2f}x')
    print(f'  Test R2:               {test_r2:.4f}')
    print(f'{"="*60}')

    visualize_training(history)
    return mlp, resnet, test_ds, history

## Visualization & Error Analysis

- **`get_gradcam`** — computes a Grad-CAM heatmap by back-propagating through `backbone.layer4`.
- **`show_gradcam_grid`** — displays original poster + Grad-CAM overlay with predicted vs. actual ROI.
- **`analyze_prediction_errors`** — scatter plot, residual plot, error distribution, and Q-Q plot
  over the entire test set, plus tables of the best and worst predictions.
- **`run_visualizations`** — orchestrates all three for highest, lowest, and random predictions.

In [ ]:
def get_gradcam(mlp: nn.Module, backbone: nn.Module, img_tensor: torch.Tensor) -> np.ndarray:
    """Returns a normalized Grad-CAM heatmap (H x W, values in [0, 1]) for img_tensor."""
    target_layer = backbone.layer4
    activations, gradients = [None], [None]

    h_fwd = target_layer.register_forward_hook(lambda _, __, out: activations.__setitem__(0, out))
    h_bwd = target_layer.register_full_backward_hook(lambda _, __, g: gradients.__setitem__(0, g[0]))

    for p in backbone.parameters(): p.requires_grad_(True)
    img = img_tensor.unsqueeze(0).to(device)
    with torch.enable_grad():
        pred = mlp(torch.flatten(backbone(img), 1)).squeeze()
        backbone.zero_grad(); mlp.zero_grad()
        pred.backward()

    h_fwd.remove(); h_bwd.remove()
    for p in backbone.parameters(): p.requires_grad_(False)

    weights = gradients[0].mean(dim=[2, 3], keepdim=True)
    cam     = F.relu((weights * activations[0]).sum(dim=1)).squeeze().detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


def show_gradcam_grid(
    mlp: nn.Module, backbone: nn.Module, dataset,
    indices=None, n: int = 5, alpha_heatmap: float = 0.45, colormap: str = 'jet'
):
    """Renders a grid of (original poster | Grad-CAM overlay) rows with prediction details."""
    if indices is None:
        indices = np.random.choice(len(dataset), size=n, replace=False).tolist()
    n    = len(indices)
    cmap = plt.get_cmap(colormap)
    fig, axes = plt.subplots(n, 2, figsize=(12, 5 * n))
    if n == 1: axes = [axes]
    mlp.eval(); backbone.eval()

    to_roi = np.expm1 if USE_LOG_TARGET else (lambda x: x)

    for row, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        raw_pil  = Image.open(dataset.paths[idx]).convert('RGB')
        img_np   = np.array(raw_pil) / 255.0

        cam      = get_gradcam(mlp, backbone, img_tensor)
        cam_up   = np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize(
                raw_pil.size, resample=Image.BILINEAR)
        ) / 255.0
        overlay  = np.clip((1 - alpha_heatmap) * img_np + alpha_heatmap * cmap(cam_up)[..., :3], 0, 1)

        with torch.no_grad():
            pred_log = mlp(torch.flatten(backbone(img_tensor.unsqueeze(0).to(device)), 1)).item()
        pred_roi = to_roi(pred_log)
        true_roi = to_roi(label.item())
        pct_err  = ((pred_roi - true_roi) / abs(true_roi)) * 100 if true_roi != 0 else 0.0

        axes[row][0].imshow(img_np)
        axes[row][0].set_title(
            f"{dataset.titles[idx]}\n"
            f"Predicted ROI: {pred_roi:>8.2f}\n"
            f"Actual ROI:    {true_roi:>8.2f}\n"
            f"Error:         {pct_err:>+7.1f}%",
            fontsize=10, family='monospace', loc='left'
        )
        axes[row][0].axis('off')

        axes[row][1].imshow(overlay)
        sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0, 1))
        plt.colorbar(sm, ax=axes[row][1], fraction=0.046, pad=0.04, label='Attention')
        axes[row][1].set_title('Grad-CAM Heatmap', fontsize=10)
        axes[row][1].axis('off')

    plt.suptitle('ResNet50 Grad-CAM  |  ROI Prediction from Poster Only', fontsize=14, y=0.995)
    plt.tight_layout()
    plt.show()


def analyze_prediction_errors(mlp: nn.Module, backbone: nn.Module, dataset):
    """Scatter, residual, error distribution, and Q-Q plots over the full test set."""
    mlp.eval(); backbone.eval()
    y_true, y_pred, titles = [], [], []
    with torch.no_grad():
        for idx in range(len(dataset)):
            img_t, label = dataset[idx]
            feat = torch.flatten(backbone(img_t.unsqueeze(0).to(device)), 1)
            y_pred.append(mlp(feat).item())
            y_true.append(label.item())
            titles.append(dataset.titles[idx])

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    errors     = y_pred - y_true
    abs_errors = np.abs(errors)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Predicted vs Actual
    ax  = axes[0, 0]
    r2  = r2_score(y_true, y_pred)
    corr, p_val = stats.pearsonr(y_true, y_pred)
    lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.5, linewidth=2, label='Perfect prediction')
    ax.scatter(y_true, y_pred, alpha=0.6, s=50)
    ax.set_xlabel('Actual log(ROI)', fontsize=12)
    ax.set_ylabel('Predicted log(ROI)', fontsize=12)
    ax.set_title(f'Predicted vs Actual\nR2={r2:.4f}, Corr={corr:.4f} (p={p_val:.2e})', fontsize=14)
    ax.legend(); ax.grid(True, alpha=0.3)
    mae  = np.mean(abs_errors)
    rmse = np.sqrt(np.mean(errors ** 2))
    ax.text(0.05, 0.95, f'MAE: {mae:.4f}\nRMSE: {rmse:.4f}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Residual plot
    ax = axes[0, 1]
    ax.scatter(y_pred, errors, alpha=0.6, s=50)
    ax.axhline(0, color='r', linestyle='--', linewidth=2)
    z = np.polyfit(y_pred, errors, 1)
    ax.plot(sorted(y_pred), np.poly1d(z)(sorted(y_pred)), 'g--', alpha=0.8, linewidth=2,
            label=f'Trend: y={z[0]:.3f}x+{z[1]:.3f}')
    ax.set_xlabel('Predicted log(ROI)', fontsize=12)
    ax.set_ylabel('Residual (Predicted - Actual)', fontsize=12)
    ax.set_title('Residual Plot', fontsize=14)
    ax.legend(); ax.grid(True, alpha=0.3)

    # Error distribution
    ax = axes[1, 0]
    ax.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    ax.axvline(0,                color='r',      linestyle='--', linewidth=2, label='Zero error')
    ax.axvline(np.mean(errors),  color='g',      linestyle='--', linewidth=2, label=f'Mean: {np.mean(errors):.3f}')
    ax.axvline(np.median(errors),color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(errors):.3f}')
    ax.set_xlabel('Error (Predicted - Actual)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Error Distribution', fontsize=14)
    ax.legend(); ax.grid(True, alpha=0.3)

    # Q-Q plot
    stats.probplot(errors, dist='norm', plot=axes[1, 1])
    axes[1, 1].set_title('Q-Q Plot', fontsize=14)
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('roi_prediction_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

    to_roi = np.expm1 if USE_LOG_TARGET else (lambda x: x)
    print(f'\n{"="*60}')
    print('ROI PREDICTION ERROR ANALYSIS')
    print(f'{"="*60}')
    print(f'  MAE  (log scale): {mae:.4f}')
    print(f'  RMSE (log scale): {rmse:.4f}')
    print(f'  R2:               {r2:.4f}')
    print(f'  Correlation:      {corr:.4f} (p={p_val:.2e})')

    for label, idx_list in [
        ('Worst 10 predictions', np.argsort(abs_errors)[-10:][::-1]),
        ('Best 10 predictions',  np.argsort(abs_errors)[:10]),
    ]:
        print(f'\n  {label}:')
        for i, idx in enumerate(idx_list, 1):
            pred_roi = to_roi(y_pred[idx]); true_roi = to_roi(y_true[idx])
            pct_err  = ((pred_roi - true_roi) / abs(true_roi)) * 100 if true_roi != 0 else 0.0
            print(f'  {i:2d}. {titles[idx]:40s} | Actual: {y_true[idx]:6.2f} | Pred: {y_pred[idx]:6.2f} | Error: {pct_err:+7.1f}%')
    print('=' * 60)


def run_visualizations(mlp: nn.Module, resnet: nn.Module, test_ds, n: int = 5):
    """Runs error analysis, then Grad-CAM grids for highest, lowest, and random predictions."""
    print('\n--- Visualization & Error Analysis ---')
    analyze_prediction_errors(mlp, resnet, test_ds)

    with torch.no_grad():
        all_preds = np.array([
            mlp(torch.flatten(resnet(img_t.unsqueeze(0).to(device)), 1)).item()
            for img_t, _ in test_ds
        ])

    print('\nHighest predicted ROI:')
    show_gradcam_grid(mlp, resnet, test_ds, indices=np.argsort(all_preds)[-n:][::-1].tolist())

    print('\nLowest predicted ROI:')
    show_gradcam_grid(mlp, resnet, test_ds, indices=np.argsort(all_preds)[:n].tolist())

    print('\nRandom samples:')
    show_gradcam_grid(mlp, resnet, test_ds,
                      indices=np.random.choice(len(all_preds), size=n, replace=False).tolist())

## Run

Executes all three stages in sequence. Re-run individual cells above to change
hyperparameters or swap datasets without restarting the runtime.

In [ ]:
train_df, test_df            = prepare_data()
mlp, resnet, test_ds, history = run_training_pipeline()
run_visualizations(mlp, resnet, test_ds, n=10)